# PyPSA-Iberia Teaching Notebook

This notebook is designed as a guided tour of what PyPSA-Eur gives us for power-system modelling students.

It combines three perspectives:
- **compiled input data**: demand, geography, renewable profiles, existing grid, existing assets
- **optimisation outputs**: build-out, dispatch, storage use, prices, curtailment, transmission use
- **interpretation**: what these results mean and which modelling questions they answer

Maps are only used where geography matters. For costs, prices over time, dispatch, and technology comparisons, tables and time-series plots are more informative.


In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import seaborn as sns
from IPython.display import display

plt.style.use("bmh")
sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "results").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


ROOT = find_repo_root()
PRIMARY_BUS_CARRIER = "AC"

# Edit these candidate paths when a new run is available.
EXPANSION_CANDIDATES = [
    ROOT / "results/iberia-v0-default/networks/base_s_9_elec_Ept80.nc",
]
CURRENT_CANDIDATES = [
    ROOT / "results/iberia-dispatch-2025/networks/base_s_9_elec_Ept80.nc",
]


def pick_first_existing(candidates: list[Path]) -> Path | None:
    for path in candidates:
        if path.exists():
            return path
    return None


expansion_path = pick_first_existing(EXPANSION_CANDIDATES)
current_path = pick_first_existing(CURRENT_CANDIDATES)

if expansion_path is None:
    raise FileNotFoundError(
        "No expansion network found. Update EXPANSION_CANDIDATES in the setup cell."
    )

n_exp = pypsa.Network(expansion_path)
n_cur = pypsa.Network(current_path) if current_path else None


def scenario_name(path: Path | None) -> str:
    return path.parents[1].name if path else "not available"


def iberia_boundaries(n: pypsa.Network, pad: float = 0.6) -> tuple[float, float, float, float]:
    return (
        float(n.buses.x.min() - pad),
        float(n.buses.x.max() + pad),
        float(n.buses.y.min() - pad),
        float(n.buses.y.max() + pad),
    )


def annotate_buses(ax, n: pypsa.Network, series: pd.Series, formatter, *, dx: float = 0.08, dy: float = 0.08, min_abs: float = 0.0, fontsize: int = 8):
    series = series.dropna()
    for bus, value in series.items():
        if abs(value) < min_abs or bus not in n.buses.index:
            continue
        row = n.buses.loc[bus]
        ax.text(
            row.x + dx,
            row.y + dy,
            formatter(bus, value),
            transform=ccrs.PlateCarree(),
            fontsize=fontsize,
            ha="left",
            va="bottom",
            bbox=dict(facecolor="white", alpha=0.78, edgecolor="none", pad=1.0),
        )


def annotate_lines(ax, n: pypsa.Network, series: pd.Series, formatter, *, min_abs: float = 0.0, fontsize: int = 7):
    series = series.dropna()
    for line, value in series.items():
        if abs(value) < min_abs or line not in n.lines.index:
            continue
        row = n.lines.loc[line]
        x0, y0 = n.buses.loc[row.bus0, ["x", "y"]]
        x1, y1 = n.buses.loc[row.bus1, ["x", "y"]]
        ax.text(
            (x0 + x1) / 2,
            (y0 + y1) / 2,
            formatter(line, value),
            transform=ccrs.PlateCarree(),
            fontsize=fontsize,
            color="darkred",
            ha="center",
            va="center",
            bbox=dict(facecolor="white", alpha=0.70, edgecolor="none", pad=0.8),
        )


def bus_loads(n: pypsa.Network) -> pd.DataFrame:
    load = n.loads_t.p_set.T.groupby(n.loads.bus).sum().T
    return pd.DataFrame(
        {
            "mean_mw": load.mean(),
            "peak_gw": load.max() / 1e3,
            "annual_twh": load.sum() / 1e6,
        }
    )


def generator_bus_capacity(n: pypsa.Network, carrier: str, column: str) -> pd.Series:
    gens = n.generators[n.generators.carrier == carrier]
    if gens.empty:
        return pd.Series(dtype=float)
    return gens.groupby("bus")[column].sum().reindex(n.buses.index, fill_value=0.0)


def availability_cf_by_bus(n: pypsa.Network, carrier: str) -> pd.Series:
    gens = n.generators[n.generators.carrier == carrier]
    if gens.empty:
        return pd.Series(dtype=float)
    return n.generators_t.p_max_pu[gens.index].mean().groupby(gens.bus).mean().reindex(n.buses.index)


def dispatched_cf_by_bus(n: pypsa.Network, carrier: str) -> pd.Series:
    gens = n.generators[n.generators.carrier == carrier]
    if gens.empty:
        return pd.Series(dtype=float)
    cf = (n.generators_t.p[gens.index].mean() / gens.p_nom_opt.replace(0.0, np.nan)).groupby(gens.bus).mean()
    return cf.reindex(n.buses.index)


def mean_ac_prices(n: pypsa.Network) -> pd.Series:
    buses = n.buses.index[n.buses.carrier == PRIMARY_BUS_CARRIER]
    return n.buses_t.marginal_price[buses].mean().sort_values(ascending=False)


def monthly_mean_price(n: pypsa.Network) -> pd.Series:
    buses = n.buses.index[n.buses.carrier == PRIMARY_BUS_CARRIER]
    return n.buses_t.marginal_price[buses].mean(axis=1).groupby(n.snapshots.month).mean()


def top_level_capacity_table(n: pypsa.Network) -> pd.DataFrame:
    installed = n.statistics.installed_capacity().groupby("carrier").sum() / 1e3
    optimal = n.statistics.optimal_capacity().groupby("carrier").sum() / 1e3
    df = pd.concat([installed.rename("installed_gw"), optimal.rename("optimal_gw")], axis=1).fillna(0.0)
    df["added_gw"] = df["optimal_gw"] - df["installed_gw"]
    return df.sort_values("optimal_gw", ascending=False)


print(f"Repository root: {ROOT}")
print(f"Expansion scenario: {scenario_name(expansion_path)} -> {expansion_path}")
print(f"Current scenario:   {scenario_name(current_path)} -> {current_path if current_path else 'not found'}")


In [ ]:
def overview_row(label: str, n: pypsa.Network, path: Path | None) -> dict:
    return {
        "scenario": label,
        "run": scenario_name(path),
        "snapshots": len(n.snapshots),
        "buses": len(n.buses),
        "lines": len(n.lines),
        "generators": len(n.generators),
        "storage_units": len(n.storage_units),
        "links": len(n.links),
        "annual_load_twh": n.loads_t.p_set.sum().sum() / 1e6,
        "mean_price_eur_per_mwh": mean_ac_prices(n).mean(),
    }


rows = [overview_row("expansion", n_exp, expansion_path)]
if n_cur is not None:
    rows.insert(0, overview_row("current", n_cur, current_path))

overview = pd.DataFrame(rows).set_index("scenario")
display(overview)

print("Generator carriers in expansion case:")
display((n_exp.generators.carrier.value_counts().rename("count")).to_frame())

print("Storage carriers in expansion case:")
display((n_exp.storage_units.carrier.value_counts().rename("count")).to_frame())


## 1. Geography, clustering, demand, and the existing grid

Start with the physical system students are modelling: the clustered buses, the AC transmission grid, and how demand is distributed in space.

This is one of the most useful maps to annotate heavily, because it connects geography to every later result.


In [ ]:
n_base = n_cur if n_cur is not None else n_exp
loads = bus_loads(n_base)
bus_size = 0.08 + np.sqrt(loads["mean_mw"]) / 180
line_capacity = (n_base.lines.s_nom_opt if "s_nom_opt" in n_base.lines else n_base.lines.s_nom) / 1e3
line_width = 0.5 + 5 * (line_capacity / line_capacity.max()) ** 1.5

fig, ax = plt.subplots(figsize=(11, 8.5), subplot_kw={"projection": ccrs.PlateCarree()})
n_base.plot.map(
    ax=ax,
    bus_size=bus_size,
    bus_color="steelblue",
    bus_alpha=0.82,
    line_width=line_width,
    line_color="dimgray",
    line_alpha=0.8,
    boundaries=iberia_boundaries(n_base),
    geomap=True,
    geomap_resolution="10m",
)

annotate_buses(
    ax,
    n_base,
    loads["peak_gw"],
    lambda bus, value: f"{bus}\npeak {value:.1f} GW\nannual {loads.at[bus, 'annual_twh']:.1f} TWh",
    min_abs=0.1,
)
annotate_lines(ax, n_base, line_capacity, lambda line, value: f"{value:.1f} GW", min_abs=0.5)

ax.set_title("Existing network topology with nodal demand and corridor capacities")
plt.show()


## 2. Resource quality and demand patterns

Students should see that PyPSA-Eur is not just an optimisation shell. It already compiles the weather-driven availability time series that explain why investment shifts geographically.

The next two maps use **available** capacity factors, not dispatched ones. That is the right place to start the discussion of resource quality before introducing curtailment and congestion.


In [ ]:
resource_carriers = [
    ("solar", "goldenrod", "Solar availability factor"),
    ("onwind", "seagreen", "Onshore wind availability factor"),
]

fig, axes = plt.subplots(
    1,
    len(resource_carriers),
    figsize=(14, 6),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
if len(resource_carriers) == 1:
    axes = [axes]

for ax, (carrier, color, title) in zip(axes, resource_carriers):
    cf = availability_cf_by_bus(n_exp, carrier)
    size = 0.15 + 0.65 * (cf.fillna(0) / cf.max())
    n_exp.plot.map(
        ax=ax,
        bus_size=size,
        bus_color=color,
        bus_alpha=0.75,
        line_width=0.6,
        line_color="lightgray",
        boundaries=iberia_boundaries(n_exp),
        geomap=True,
    )
    annotate_buses(ax, n_exp, cf, lambda bus, value: f"{bus}\nCF {value:.3f}", min_abs=0.01)
    ax.set_title(title)

plt.show()


A key teaching question is whether **diversity in location and technology** reduces variability.

There is no single correct correlation metric, so this section separates several different ideas:
- **annual hourly correlation**: captures the full seasonal and diurnal structure over the year
- **daily-energy correlation**: asks whether sunny days also tend to be windy days at a given location
- **average within-day correlation**: for each day, correlate the 24-hour solar and wind shapes, then average across days
- **anomaly correlation**: remove the regular seasonal and hour-of-day pattern first, then correlate the remaining weather-driven deviations
- **low-output coincidence**: correlation can still miss the event students often care about most, namely how often low-renewable conditions happen together

These metrics answer different questions. A high annual correlation may mostly reflect summer-vs-winter structure, while anomaly correlation is closer to the question of whether the same weather systems drive shortages in multiple places at once.


In [ ]:
def availability_profiles_by_bus(n: pypsa.Network, carrier: str, weight_col: str = "p_nom_opt") -> pd.DataFrame:
    gens = n.generators[n.generators.carrier == carrier]
    if gens.empty:
        return pd.DataFrame(index=n.snapshots)

    weights = gens.get(weight_col, pd.Series(index=gens.index, dtype=float)).fillna(0.0)
    weights = weights.where(weights > 0.0, gens.get("p_nom", 0.0)).fillna(0.0)
    weights = weights.where(weights > 0.0, 1.0)

    profiles = n.generators_t.p_max_pu[gens.index].mul(weights, axis=1)
    bus_weights = weights.groupby(gens.bus).sum()
    bus_profiles = profiles.T.groupby(gens.bus).sum().T.div(bus_weights, axis=1)
    return bus_profiles.reindex(sorted(bus_profiles.columns), axis=1)


def daily_mean_profiles(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby(df.index.date).mean()


def mean_daily_shape_correlation(a: pd.Series, b: pd.Series) -> float:
    daily_corrs = []
    for _, a_day in a.groupby(a.index.date):
        b_day = b.loc[a_day.index]
        if a_day.std() == 0 or b_day.std() == 0:
            continue
        daily_corrs.append(a_day.corr(b_day))
    return float(np.mean(daily_corrs)) if daily_corrs else np.nan


def remove_month_hour_climatology(df: pd.DataFrame) -> pd.DataFrame:
    keys = [df.index.month, df.index.hour]
    climatology = df.groupby(keys).transform("mean")
    return df - climatology


def low_output_share(a: pd.Series, b: pd.Series, quantile: float = 0.2) -> float:
    a_threshold = a.quantile(quantile)
    b_threshold = b.quantile(quantile)
    return ((a <= a_threshold) & (b <= b_threshold)).mean()


solar_profiles = availability_profiles_by_bus(n_exp, "solar")
wind_profiles = availability_profiles_by_bus(n_exp, "onwind")
common_buses = solar_profiles.columns.intersection(wind_profiles.columns)

solar_daily = daily_mean_profiles(solar_profiles[common_buses])
wind_daily = daily_mean_profiles(wind_profiles[common_buses])
solar_anom = remove_month_hour_climatology(solar_profiles[common_buses])
wind_anom = remove_month_hour_climatology(wind_profiles[common_buses])

complementarity = pd.DataFrame(index=common_buses)
complementarity["annual_hourly_corr"] = [solar_profiles[bus].corr(wind_profiles[bus]) for bus in common_buses]
complementarity["daily_energy_corr"] = [solar_daily[bus].corr(wind_daily[bus]) for bus in common_buses]
complementarity["mean_within_day_corr"] = [mean_daily_shape_correlation(solar_profiles[bus], wind_profiles[bus]) for bus in common_buses]
complementarity["anomaly_corr"] = [solar_anom[bus].corr(wind_anom[bus]) for bus in common_buses]
complementarity["joint_low_output_share_q20"] = [low_output_share(solar_profiles[bus], wind_profiles[bus], 0.2) for bus in common_buses]
display(complementarity.sort_values("annual_hourly_corr"))

fig, ax = plt.subplots(figsize=(10.5, 8), subplot_kw={"projection": ccrs.PlateCarree()})
color_values = complementarity["anomaly_corr"].reindex(n_exp.buses.index)
norm = plt.Normalize(-1, 1)
cmap = plt.cm.RdBu_r
bus_colors = pd.Series(
    [cmap(norm(v)) if pd.notna(v) else (0.8, 0.8, 0.8, 1.0) for v in color_values],
    index=n_exp.buses.index,
)
bus_sizes = 0.12 + 0.35 * complementarity["joint_low_output_share_q20"].reindex(n_exp.buses.index).fillna(0.0) / max(complementarity["joint_low_output_share_q20"].max(), 1e-6)

n_exp.plot.map(
    ax=ax,
    bus_size=bus_sizes,
    bus_color=bus_colors,
    bus_alpha=0.9,
    line_width=0.7,
    line_color="lightgray",
    boundaries=iberia_boundaries(n_exp),
    geomap=True,
)
annotate_buses(
    ax,
    n_exp,
    complementarity["anomaly_corr"],
    lambda bus, value: (
        f"{bus}\nyr {value:+.2f}"
        f"\nday {complementarity.at[bus, 'daily_energy_corr']:+.2f}"
        f"\n24h {complementarity.at[bus, 'mean_within_day_corr']:+.2f}"
        f"\nanom {complementarity.at[bus, 'anomaly_corr']:+.2f}"
        f"\nlow20 {100 * complementarity.at[bus, 'joint_low_output_share_q20']:.1f}%"
    ),
    min_abs=0.0,
)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, orientation="horizontal", pad=0.03, aspect=45, label="Solar-wind anomaly correlation after removing month-hour climatology")
ax.set_title("Solar-wind complementarity by node, with anomaly correlation and joint low-output frequency")
plt.show()


In [ ]:
solar_hourly_corr = solar_profiles.corr()
wind_hourly_corr = wind_profiles.corr()
solar_anom_corr = solar_anom.corr()
wind_anom_corr = wind_anom.corr()
solar_wind_daily_corr = pd.DataFrame(
    {solar_bus: wind_daily.corrwith(solar_daily[solar_bus]) for solar_bus in solar_daily.columns}
).T

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
sns.heatmap(solar_hourly_corr, vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f", ax=axes[0, 0])
axes[0, 0].set_title("Solar-solar hourly correlation across nodes")

sns.heatmap(wind_hourly_corr, vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f", ax=axes[0, 1])
axes[0, 1].set_title("Wind-wind hourly correlation across nodes")

sns.heatmap(solar_anom_corr, vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f", ax=axes[1, 0])
axes[1, 0].set_title("Solar-solar anomaly correlation across nodes")

sns.heatmap(wind_anom_corr, vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f", ax=axes[1, 1])
axes[1, 1].set_title("Wind-wind anomaly correlation across nodes")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7.8, 6.2))
sns.heatmap(solar_wind_daily_corr, vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f", ax=ax)
ax.set_title("Daily-energy correlation: solar node vs wind node")
ax.set_xlabel("wind node")
ax.set_ylabel("solar node")

plt.tight_layout()
plt.show()


Correlation is still not the whole story. System planners often care about **coincidence of bad conditions** more than average covariance.

The next view asks how often solar and wind are simultaneously low, using each series' own 20th percentile as the threshold. It then compares that coincidence risk for single-node and diversified portfolios.


In [ ]:
def normalize_to_mean(df: pd.DataFrame) -> pd.DataFrame:
    return df.div(df.mean()).replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="any")


solar_norm = normalize_to_mean(solar_profiles[common_buses])
wind_norm = normalize_to_mean(wind_profiles[common_buses])

joint_low_matrix = pd.DataFrame(
    {solar_bus: [low_output_share(solar_profiles[solar_bus], wind_profiles[wind_bus], 0.2) for wind_bus in common_buses] for solar_bus in common_buses},
    index=common_buses,
).T
joint_low_matrix.columns = common_buses

portfolio_low_output = pd.Series(
    {
        "average single solar node": pd.Series([(solar_norm[col] <= solar_norm[col].quantile(0.2)).mean() for col in solar_norm.columns]).mean(),
        "diversified solar portfolio": (solar_norm.mean(axis=1) <= solar_norm.mean(axis=1).quantile(0.2)).mean(),
        "average single wind node": pd.Series([(wind_norm[col] <= wind_norm[col].quantile(0.2)).mean() for col in wind_norm.columns]).mean(),
        "diversified wind portfolio": (wind_norm.mean(axis=1) <= wind_norm.mean(axis=1).quantile(0.2)).mean(),
        "average co-located solar+wind mix": pd.Series([
            low_output_share(solar_norm[bus], wind_norm[bus], 0.2) for bus in common_buses
        ]).mean(),
        "fully diversified solar+wind portfolio": low_output_share(
            solar_norm.mean(axis=1), wind_norm.mean(axis=1), 0.2
        ),
    }
).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
sns.heatmap(100 * joint_low_matrix, cmap="magma_r", annot=True, fmt=".1f", ax=axes[0])
axes[0].set_title("Joint low-output hours [%]: solar node vs wind node")
axes[0].set_xlabel("wind node")
axes[0].set_ylabel("solar node")

(100 * portfolio_low_output).plot.barh(ax=axes[1], color="firebrick")
axes[1].set_title("Coincidence of low-output conditions")
axes[1].set_xlabel("share of hours [%]")

plt.tight_layout()
plt.show()


In [ ]:
def normalize_to_mean(df: pd.DataFrame) -> pd.DataFrame:
    return df.div(df.mean()).replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="any")


def profile_metrics(s: pd.Series) -> pd.Series:
    return pd.Series(
        {
            "std_of_mean_normalized_output": s.std(),
            "p05_of_mean_normalized_output": s.quantile(0.05),
            "share_below_half_mean": (s < 0.5).mean(),
            "std_of_hourly_ramp": s.diff().std(),
        }
    )


solar_norm = normalize_to_mean(solar_profiles[common_buses])
wind_norm = normalize_to_mean(wind_profiles[common_buses])
mixed_norm = pd.concat(
    [solar_norm.add_prefix("solar_"), wind_norm.add_prefix("wind_")],
    axis=1,
)

portfolio_cases = {
    "average single solar node": pd.DataFrame([profile_metrics(solar_norm[col]) for col in solar_norm.columns]).mean(),
    "diversified solar portfolio": profile_metrics(solar_norm.mean(axis=1)),
    "average single wind node": pd.DataFrame([profile_metrics(wind_norm[col]) for col in wind_norm.columns]).mean(),
    "diversified wind portfolio": profile_metrics(wind_norm.mean(axis=1)),
    "average co-located solar+wind mix": pd.DataFrame([profile_metrics(0.5 * (solar_norm[bus] + wind_norm[bus])) for bus in common_buses]).mean(),
    "fully diversified solar+wind portfolio": profile_metrics(mixed_norm.mean(axis=1)),
}
portfolio_table = pd.DataFrame(portfolio_cases).T
display(portfolio_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
portfolio_table["std_of_mean_normalized_output"].sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Variability of normalized output")
axes[0].set_xlabel("standard deviation, lower is better")

portfolio_table["p05_of_mean_normalized_output"].sort_values().plot.barh(ax=axes[1], color="darkorange")
axes[1].set_title("Low-output resilience of normalized output")
axes[1].set_xlabel("5th percentile, higher is better")

plt.tight_layout()
plt.show()


Another useful question is whether renewable **technical potential** is binding at particular nodes.

In PyPSA-Eur, the renewable profile-building workflow first computes a spatial availability matrix using land-use or sea-area exclusions, then converts that into a node-level maximal installable capacity `p_nom_max`. For onshore technologies this is closely related to available land; for offshore technologies it reflects sea-area and distance constraints.

So the operational question in the solved network is: how much of each node's available potential does the optimiser actually use?


In [ ]:
def potential_table(n: pypsa.Network, carrier: str) -> pd.DataFrame:
    gens = n.generators[n.generators.carrier == carrier]
    if gens.empty or "p_nom_max" not in gens.columns:
        return pd.DataFrame(columns=["optimal_gw", "potential_gw", "utilisation_of_potential", "binding"])
    df = gens.groupby("bus")[["p_nom_opt", "p_nom_max"]].sum() / 1e3
    df = df.rename(columns={"p_nom_opt": "optimal_gw", "p_nom_max": "potential_gw"})
    df["utilisation_of_potential"] = df["optimal_gw"] / df["potential_gw"].replace(0.0, np.nan)
    df["binding"] = df["utilisation_of_potential"] >= 0.95
    return df.reindex(n.buses.index)


solar_potential = potential_table(n_exp, "solar")
wind_potential = potential_table(n_exp, "onwind")
potential_summary = pd.concat(
    {
        "solar": solar_potential,
        "onwind": wind_potential,
    },
    axis=1,
)
display(potential_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.3), subplot_kw={"projection": ccrs.PlateCarree()})
for ax, carrier, title, color in [
    (axes[0], "solar", "Solar potential and usage", "goldenrod"),
    (axes[1], "onwind", "Onshore wind potential and usage", "seagreen"),
]:
    table = potential_table(n_exp, carrier)
    sizes = 0.05 + table["potential_gw"].fillna(0.0) / max(table["potential_gw"].max(), 1e-6)
    colors = table["utilisation_of_potential"].reindex(n_exp.buses.index)
    cmap = plt.cm.YlOrRd
    norm = plt.Normalize(0, min(1.0, np.nanmax(colors.fillna(0.0)) if len(colors.dropna()) else 1.0))
    bus_colors = pd.Series([cmap(norm(v)) if pd.notna(v) else (0.85, 0.85, 0.85, 1.0) for v in colors], index=n_exp.buses.index)
    n_exp.plot.map(
        ax=ax,
        bus_size=sizes,
        bus_color=bus_colors,
        line_width=0.6,
        line_color="lightgray",
        boundaries=iberia_boundaries(n_exp),
        geomap=True,
    )
    annotate_buses(
        ax,
        n_exp,
        table["potential_gw"],
        lambda bus, value: f"{bus}\n{table.at[bus, 'optimal_gw']:.1f}/{value:.1f} GW\nuse {100 * table.at[bus, 'utilisation_of_potential']:.1f}%",
        min_abs=0.5 if carrier == "solar" else 0.01,
    )
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, orientation="horizontal", pad=0.03, aspect=35, label="share of node potential used")
    ax.set_title(title)

plt.tight_layout()
plt.show()

binding_summary = pd.concat(
    {
        "solar": solar_potential[["optimal_gw", "potential_gw", "utilisation_of_potential", "binding"]],
        "onwind": wind_potential[["optimal_gw", "potential_gw", "utilisation_of_potential", "binding"]],
    },
    axis=1,
)
display(binding_summary)


In [ ]:
carrier_binding = pd.DataFrame(
    {
        "optimal_gw": [solar_potential["optimal_gw"].sum(), wind_potential["optimal_gw"].sum()],
        "potential_gw": [solar_potential["potential_gw"].sum(), wind_potential["potential_gw"].sum()],
    },
    index=["solar", "onwind"],
)
carrier_binding["utilisation_of_potential"] = carrier_binding["optimal_gw"] / carrier_binding["potential_gw"]
carrier_binding["number_of_binding_nodes"] = [int(solar_potential["binding"].fillna(False).sum()), int(wind_potential["binding"].fillna(False).sum())]
display(carrier_binding)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
carrier_binding["utilisation_of_potential"].mul(100).plot.bar(ax=axes[0], color=["goldenrod", "seagreen"])
axes[0].set_title("System-wide use of node-level technical potential")
axes[0].set_ylabel("% of total potential used")

carrier_binding["number_of_binding_nodes"].plot.bar(ax=axes[1], color=["goldenrod", "seagreen"])
axes[1].set_title("Nodes where potential is effectively binding")
axes[1].set_ylabel("count of nodes")

plt.tight_layout()
plt.show()

print(
    f"Solar uses {100 * carrier_binding.at['solar', 'utilisation_of_potential']:.1f}% of total modeled node potential "
    f"across {int(carrier_binding.at['solar', 'number_of_binding_nodes'])} effectively binding nodes."
)
print(
    f"Onshore wind uses {100 * carrier_binding.at['onwind', 'utilisation_of_potential']:.1f}% of total modeled node potential "
    f"across {int(carrier_binding.at['onwind', 'number_of_binding_nodes'])} effectively binding nodes."
)


In [ ]:
load = n_exp.loads_t.p_set.sum(axis=1) / 1e3
week = load.iloc[:168].values.reshape(7, 24)
monthly_load = load.groupby(load.index.month).mean()
duration = load.sort_values(ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

axes[0].plot(monthly_load.index, monthly_load.values, marker="o", color="darkred")
axes[0].set_title("Average system load by month")
axes[0].set_xlabel("month")
axes[0].set_ylabel("GW")

axes[1].plot(duration.values, color="navy")
axes[1].set_title("Load duration curve")
axes[1].set_xlabel("hour rank")
axes[1].set_ylabel("GW")

sns.heatmap(
    week,
    cmap="YlOrRd",
    ax=axes[2],
    xticklabels=range(24),
    yticklabels=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
)
axes[2].set_title("One representative week of demand")
axes[2].set_xlabel("hour of day")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()


## 3. Existing assets versus the expansion solution

This is the core planning question.

There are two useful comparisons here:
- **installed vs optimal** inside the expansion run
- **current scenario vs expansion scenario** if both solved networks are available

The maps below stay focused on technologies where geography is central to the story.


In [ ]:
cap_exp = top_level_capacity_table(n_exp)
if n_cur is not None:
    cap_cur = n_cur.statistics.optimal_capacity().groupby("carrier").sum() / 1e3
    cap_comp = pd.concat(
        [
            cap_cur.rename("current_gw"),
            cap_exp["installed_gw"].rename("installed_in_expansion_gw"),
            cap_exp["optimal_gw"].rename("expansion_gw"),
        ],
        axis=1,
    ).fillna(0.0)
else:
    cap_comp = cap_exp.rename(columns={"installed_gw": "current_gw", "optimal_gw": "expansion_gw"})

display(cap_comp.sort_values(cap_comp.columns[-1], ascending=False).head(15))

top = cap_comp.sort_values(cap_comp.columns[-1], ascending=False).head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5.8))
y = np.arange(len(top.index))
ax.barh(y - 0.18, top.iloc[:, 0], height=0.34, label=top.columns[0], color="slategray")
if top.shape[1] > 1:
    ax.barh(y + 0.18, top.iloc[:, -1], height=0.34, label=top.columns[-1], color="darkorange")
ax.set_yticks(y)
ax.set_yticklabels(top.index)
ax.set_xlabel("GW")
ax.set_title("Technology capacities: current versus expansion")
ax.legend()
plt.show()


In [ ]:
tech_maps = [
    ("solar", "darkorange", "gold", "Solar: existing (inner) and total optimal (outer)"),
    ("onwind", "darkgreen", "mediumseagreen", "Onshore wind: existing/current (inner) and total optimal (outer)"),
]

fig, axes = plt.subplots(
    1,
    len(tech_maps),
    figsize=(15, 6.5),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
if len(tech_maps) == 1:
    axes = [axes]

for ax, (carrier, inner_color, outer_color, title) in zip(axes, tech_maps):
    existing = generator_bus_capacity(n_exp, carrier, "p_nom") / 1e3
    optimal = generator_bus_capacity(n_exp, carrier, "p_nom_opt") / 1e3
    added = (optimal - existing).clip(lower=0.0)

    n_exp.plot.map(
        ax=ax,
        bus_size=0.05 + optimal / max(optimal.max(), 1e-6),
        bus_color=outer_color,
        bus_alpha=0.45,
        line_width=0.8,
        line_color="lightgray",
        boundaries=iberia_boundaries(n_exp),
        geomap=True,
    )
    n_exp.plot.map(
        ax=ax,
        bus_size=existing / max(optimal.max(), 1e-6),
        bus_color=inner_color,
        bus_alpha=0.95,
        line_width=0,
        boundaries=iberia_boundaries(n_exp),
        geomap=False,
    )
    annotate_buses(
        ax,
        n_exp,
        optimal,
        lambda bus, value: f"{bus}\n{existing.get(bus, 0.0):.1f} + {added.get(bus, 0.0):.1f} GW",
        min_abs=0.5,
    )
    ax.set_title(title)

plt.show()


In [ ]:
existing_lines = (n_cur.lines.s_nom_opt if n_cur is not None else n_exp.lines.s_nom) / 1e3
optimal_lines = n_exp.lines.s_nom_opt / 1e3
added_lines = (optimal_lines - existing_lines).clip(lower=0.0)

fig, ax = plt.subplots(figsize=(11, 8), subplot_kw={"projection": ccrs.PlateCarree()})
n_exp.plot.map(
    ax=ax,
    bus_size=0.12,
    bus_color="steelblue",
    bus_alpha=0.85,
    line_width=0.6 + 4 * (optimal_lines / optimal_lines.max()) ** 1.3,
    line_color="dimgray",
    boundaries=iberia_boundaries(n_exp),
    geomap=True,
)
annotate_lines(
    ax,
    n_exp,
    added_lines,
    lambda line, value: f"+{value:.1f} GW\n({existing_lines[line]:.1f}→{optimal_lines[line]:.1f})",
    min_abs=0.05,
)
ax.set_title("Transmission reinforcement in the expansion solution")
plt.show()

line_table = pd.DataFrame(
    {
        "existing_gw": existing_lines,
        "optimal_gw": optimal_lines,
        "added_gw": added_lines,
    }
).sort_values("added_gw", ascending=False)
display(line_table.head(10))


## 4. How the expansion system operates

Capacity is only part of the story. Students should also see how the optimised system actually serves demand through the year.

The first plot shows annual energy contributions. The second shows one winter week of dispatch, which is usually more intuitive than looking at annual averages alone.


In [ ]:
annual_energy = (n_exp.statistics.energy_balance(groupby="carrier") / 1e6).sort_values(ascending=False)
annual_energy_plot = annual_energy[annual_energy.abs() > 0.1]

gen_dispatch = n_exp.generators_t.p.T.groupby(n_exp.generators.carrier).sum().T / 1e3
store_dispatch = n_exp.storage_units_t.p.clip(lower=0).T.groupby(n_exp.storage_units.carrier).sum().T / 1e3
dispatch = pd.concat([gen_dispatch, store_dispatch], axis=1).groupby(level=0, axis=1).sum()
major = dispatch.mean().sort_values(ascending=False).head(6).index
dispatch_week = dispatch.copy()
dispatch_week["other"] = dispatch_week.drop(columns=major, errors="ignore").sum(axis=1)
dispatch_week = dispatch_week[list(major) + ["other"]]

winter_week = slice("2013-01-14", "2013-01-20")
load_week = n_exp.loads_t.p_set.sum(axis=1).loc[winter_week] / 1e3

fig, axes = plt.subplots(1, 2, figsize=(17, 5.2))

annual_energy_plot.sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Annual net energy balance by carrier")
axes[0].set_xlabel("TWh")

dispatch_week.loc[winter_week].plot.area(ax=axes[1], linewidth=0, alpha=0.9)
axes[1].plot(load_week.index, load_week.values, color="black", linewidth=2, label="load")
axes[1].set_title("Representative winter week: dispatch stack")
axes[1].set_ylabel("GW")
axes[1].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))

plt.tight_layout()
plt.show()


In [ ]:
cf_table = pd.DataFrame(
    {
        "solar_available": availability_cf_by_bus(n_exp, "solar"),
        "solar_dispatched": dispatched_cf_by_bus(n_exp, "solar"),
        "onwind_available": availability_cf_by_bus(n_exp, "onwind"),
        "onwind_dispatched": dispatched_cf_by_bus(n_exp, "onwind"),
    }
).sort_index()
display(cf_table)

duration = n_exp.snapshot_weightings.generators.sum()
curtailed_abs = n_exp.statistics.curtailment(bus_carrier=["AC", "low voltage"], aggregate_across_components=True)
available_energy = n_exp.statistics.optimal_capacity("Generator", bus_carrier=["AC", "low voltage"]) * duration
curtailment_pct = (100 * curtailed_abs / available_energy).dropna().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(cf_table, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("Available versus dispatched capacity factors by bus")
axes[0].set_xlabel("")
axes[0].set_ylabel("bus")

curtailment_pct.plot.bar(ax=axes[1], color="indianred")
axes[1].set_title("Curtailment by carrier")
axes[1].set_ylabel("% of available energy")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 5. Prices, scarcity, and locational signals

Marginal prices are one of the most important optimisation outputs to teach. They connect dispatch, congestion, scarcity, curtailment, and infrastructure needs.

The most useful combination here is:
- a time-domain comparison of prices in the current and expansion cases
- a map of average nodal prices with explicit annotations
- a technology market-value comparison


In [ ]:
prices_exp = n_exp.buses_t.marginal_price[n_exp.buses.index[n_exp.buses.carrier == PRIMARY_BUS_CARRIER]]
duration_exp = prices_exp.stack().sort_values(ascending=False).reset_index(drop=True)
monthly_exp = monthly_mean_price(n_exp)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))

axes[0].plot(duration_exp.values, label="expansion", color="darkorange")
if n_cur is not None:
    prices_cur = n_cur.buses_t.marginal_price[n_cur.buses.index[n_cur.buses.carrier == PRIMARY_BUS_CARRIER]]
    duration_cur = prices_cur.stack().sort_values(ascending=False).reset_index(drop=True)
    axes[0].plot(duration_cur.values, label="current", color="slategray")
axes[0].set_title("Price duration curve")
axes[0].set_xlabel("hour rank")
axes[0].set_ylabel("EUR/MWh")
axes[0].legend()

axes[1].plot(monthly_exp.index, monthly_exp.values, marker="o", label="expansion", color="darkorange")
if n_cur is not None:
    axes[1].plot(monthly_mean_price(n_cur).index, monthly_mean_price(n_cur).values, marker="o", label="current", color="slategray")
axes[1].set_title("Monthly average system price")
axes[1].set_xlabel("month")
axes[1].set_ylabel("EUR/MWh")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
bus_prices = mean_ac_prices(n_exp).reindex(n_exp.buses.index)
market_values = n_exp.statistics.market_value(bus_carrier="AC", aggregate_across_components=True).dropna()
market_values = market_values[~market_values.index.isin(["-", "AC"])]
market_values = market_values.sort_values(ascending=False).head(10)

fig = plt.figure(figsize=(16, 6))
gs = fig.add_gridspec(1, 2, width_ratios=[1.2, 0.9])

ax0 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
size = pd.Series(0.22, index=n_exp.buses.index)
cmap = plt.cm.YlOrRd
norm = plt.Normalize(bus_prices.min(), bus_prices.max())
colors = pd.Series([cmap(norm(v)) if pd.notna(v) else (0.8, 0.8, 0.8, 1.0) for v in bus_prices], index=n_exp.buses.index)
n_exp.plot.map(
    ax=ax0,
    bus_size=size,
    bus_color=colors,
    line_width=0.8,
    line_color="lightgray",
    boundaries=iberia_boundaries(n_exp),
    geomap=True,
)
annotate_buses(ax0, n_exp, bus_prices, lambda bus, value: f"{bus}\n{value:.1f} €/MWh", min_abs=1.0)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax0, orientation="horizontal", pad=0.03, aspect=40, label="Average AC nodal price [EUR/MWh]")
ax0.set_title("Average nodal prices in the expansion solution")

ax1 = fig.add_subplot(gs[0, 1])
market_values.sort_values().plot.barh(ax=ax1, color="mediumpurple")
ax1.set_title("Market value by technology")
ax1.set_xlabel("EUR/MWh")

plt.tight_layout()
plt.show()

display(bus_prices.sort_values(ascending=False).rename("avg_price_eur_per_mwh").to_frame())


## 6. Transmission use and congestion proxies

For students, congestion is easier to understand if they can see three things at once:
- which buses are net exporters or importers on average
- which corridors operate heavily
- whether reinforcement happens on the same corridors

Because this network is small, it is worth annotating the line statistics directly.


In [ ]:
line_loading = n_exp.lines_t.p0.abs().divide(n_exp.lines.s_nom_opt, axis=1)
line_loading_95 = line_loading.quantile(0.95)
line_loading_mean = line_loading.mean()

net_import = pd.Series(0.0, index=n_exp.buses.index)
mean_p0 = n_exp.lines_t.p0.mean()
for line, row in n_exp.lines.iterrows():
    net_import[row.bus0] -= mean_p0[line]
    net_import[row.bus1] += mean_p0[line]
net_import_gw = net_import / 1e3

fig, ax = plt.subplots(figsize=(11, 8), subplot_kw={"projection": ccrs.PlateCarree()})
bus_size = 0.08 + net_import_gw.abs() / max(net_import_gw.abs().max(), 1e-6)
bus_color = net_import_gw.map(lambda v: "firebrick" if v > 0 else "royalblue")
n_exp.plot.map(
    ax=ax,
    bus_size=bus_size,
    bus_color=bus_color,
    bus_alpha=0.82,
    line_width=0.6 + 5 * (line_loading_95 / line_loading_95.max()) ** 1.5,
    line_color="dimgray",
    boundaries=iberia_boundaries(n_exp),
    geomap=True,
)
annotate_buses(
    ax,
    n_exp,
    net_import_gw,
    lambda bus, value: f"{bus}\n{'import' if value > 0 else 'export'} {abs(value):.2f} GW",
    min_abs=0.03,
)
annotate_lines(ax, n_exp, line_loading_95, lambda line, value: f"p95 {100*value:.0f}%", min_abs=0.45)
ax.set_title("Average net positions and 95th-percentile corridor loading")
plt.show()

transmission_table = pd.DataFrame(
    {
        "mean_loading": line_loading_mean,
        "p95_loading": line_loading_95,
        "existing_gw": existing_lines,
        "optimal_gw": optimal_lines,
        "added_gw": added_lines,
    }
).sort_values("p95_loading", ascending=False)
display(transmission_table)


## 7. Costs, operating expenditure, and a compact results summary

This last section is useful for tying the physical story back to planning economics.

Students can ask:
- Which technologies drive capital expenditure?
- Which technologies drive variable system cost?
- Are price reductions coming from more capacity, better geography, more transmission, or all three?


In [ ]:
capex = (n_exp.statistics.capex().groupby("carrier").sum() / 1e9).sort_values(ascending=False)
opex = (n_exp.statistics.opex().groupby("carrier").sum() / 1e9).sort_values(ascending=False)
cost_table = pd.concat([capex.rename("capex_bn_eur_per_year"), opex.rename("opex_bn_eur_per_year")], axis=1).fillna(0.0)
cost_table = cost_table[(cost_table.abs().max(axis=1) > 0.001)].sort_values("capex_bn_eur_per_year", ascending=False)

summary = pd.Series(
    {
        "annual_load_twh": n_exp.loads_t.p_set.sum().sum() / 1e6,
        "mean_ac_price_eur_per_mwh": prices_exp.stack().mean(),
        "std_ac_price_eur_per_mwh": prices_exp.stack().std(),
        "min_ac_price_eur_per_mwh": prices_exp.stack().min(),
        "max_ac_price_eur_per_mwh": prices_exp.stack().max(),
        "share_of_hours_below_0_1_eur_per_mwh": (prices_exp < 0.1).mean().mean(),
        "total_capex_bn_eur_per_year": n_exp.statistics.capex().sum() / 1e9,
        "total_opex_bn_eur_per_year": n_exp.statistics.opex().sum() / 1e9,
        "total_added_ac_grid_gw": added_lines.sum(),
    }
).rename("value")

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
cost_table.head(10).iloc[::-1].plot.barh(ax=axes[0])
axes[0].set_title("Top capital and operating cost drivers")
axes[0].set_xlabel("billion EUR/year")

summary.sort_values().plot.barh(ax=axes[1], color="teal")
axes[1].set_title("Compact expansion-case summary")
axes[1].set_xlabel("value")

plt.tight_layout()
plt.show()

display(cost_table)
display(summary.to_frame())


## Suggested teaching prompts

Use the notebook as a narrative rather than a plot gallery.

- Why does the model place more solar in some regions than others even within Iberia?
- Why are available and dispatched capacity factors not the same?
- Why can prices fall on average while volatility increases?
- Which transmission corridors are reinforced, and what price or flow pattern explains that?
- Why does the model keep firm thermal capacity even with a large renewable build-out?
- Which results come from **input data quality** and which come from **optimisation trade-offs**?

When a new sector-coupled run is ready, the same structure can be extended by switching `PRIMARY_BUS_CARRIER` and adding carrier-specific sections for `H2`, `gas`, or `heat` using the same PyPSA statistics interface.
